# Survival-prognostic epistasis: embed & analyze

Runs the embedding pipeline on 6,146 pairs:
- **715 significant** — FDR < 0.10 survival + confounder cascade (gene burden, TMB, cancer type)
- **2,325 TCGA null** — same gene, similar distance/patients, p > 0.5
- **3,106 germline null** — germline-flagged positions in same genes

Includes coding, silent, intronic, UTR, and splice-region variants.
525 of 715 significant pairs are both-noncoding (345 intron/intron, 17 silent/silent).

**Cell 1** — Config
**Cell 2** — Load pairs & embed (run on cluster per env)
**Cell 3** — Compute cov_inv (or load existing)
**Cell 4** — Recompute metrics
**Cell 5** — Load residuals & compare groups
**Cell 6** — Statistical tests (MWU + permutation on mean)
**Cell 7** — Figures

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1: Config
# ---------------------------------------------------------------------------
import sys, os, logging
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO)

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "notebooks" / "paper_data_config.py").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.paper_data_config import EPISTASIS_PAPER_ROOT, embeddings_dir
from notebooks.processing.pipeline_config import (
    SOURCE_COL, ID_COL, SOURCE_MODEL_MAP, COV_INV_SOURCE_NAMES, get_batch_size,
)
from notebooks.processing.process_epistasis import (
    get_model_keys_for_env, ENV_PROFILES, FULL_MODEL_CONFIG,
)

FIG_DIR = EPISTASIS_PAPER_ROOT / "figures"
OUTPUT_BASE = embeddings_dir()
SURVIVAL_SOURCE = "tcga_survival"

# ── What to run ──
ENV_PROFILE = os.environ.get("CONDA_DEFAULT_ENV", "base")
MODEL_KEYS = ENV_PROFILES.get(ENV_PROFILE, list(ENV_PROFILES.get("all", [])))
# MODEL_KEYS = ["alphagenome"]  # uncomment to run single model
FORCE = False
BATCH_SIZE_OVERRIDE = None

print(f"Env: {ENV_PROFILE}")
print(f"Models: {MODEL_KEYS}")
print(f"Output: {OUTPUT_BASE}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2: Load survival pairs & embed (run on cluster per env)
# ---------------------------------------------------------------------------
from notebooks.processing.process_epistasis import run_from_single_dataframe

# Load the combined pipeline file (sig + tcga_null + germline_null)
groups_df = pd.read_csv(FIG_DIR / "tcga_survival_pipeline_all_groups.csv")
print(f"Pipeline pairs: {len(groups_df)}")
print(groups_df["group"].value_counts().to_string())

# Format for embedding pipeline
embed_df = groups_df[["epistasis_id", "gene", "dist", "group"]].copy()
embed_df[SOURCE_COL] = SURVIVAL_SOURCE
embed_df[ID_COL] = embed_df["epistasis_id"]
embed_df = embed_df.dropna(subset=[ID_COL])

batch_size_by_model = (
    {k: BATCH_SIZE_OVERRIDE for k in MODEL_KEYS}
    if BATCH_SIZE_OVERRIDE
    else {k: get_batch_size(k) for k in MODEL_KEYS}
)

print(f"\nEmbedding {len(embed_df)} pairs through {len(MODEL_KEYS)} models...")

run_from_single_dataframe(
    embed_df,
    output_base=OUTPUT_BASE,
    source_col=SOURCE_COL,
    model_keys=MODEL_KEYS,
    source_model_map=SOURCE_MODEL_MAP,
    id_col=ID_COL,
    show_progress=True,
    force=FORCE,
    batch_size=16,
    batch_size_by_model=batch_size_by_model,
    use_by_tool=True,
)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3: Compute cov_inv (or use existing from main pipeline)
# ---------------------------------------------------------------------------
from notebooks.processing.process_epistasis import compute_cov_inv

cache_dir = OUTPUT_BASE / "cache"

# Check if cov_inv already exists from the main pipeline run
existing_cov = sorted(cache_dir.glob("*_cov_inv.npz")) if cache_dir.exists() else []
if existing_cov:
    print(f"Found existing cov_inv for {len(existing_cov)} models:")
    for p in existing_cov:
        print(f"  {p.stem.replace('_cov_inv', '')}")
    print("Skipping cov_inv computation (using existing).")
else:
    print("No existing cov_inv found. Computing from COV_INV_SOURCE_NAMES...")
    # Need the null source data
    from notebooks.processing.pipeline_config import get_single_dataframe_path
    from notebooks.paper_data_config import data_dir
    single_path = get_single_dataframe_path(data_dir)
    if single_path:
        df_all = pd.read_csv(single_path, sep=None, engine="python")
        df_null = df_all[df_all[SOURCE_COL].astype(str).isin(COV_INV_SOURCE_NAMES)]
        okgp_sources = [s for s in df_all[SOURCE_COL].unique() if s in COV_INV_SOURCE_NAMES]
        model_keys_cov = [
            k for k in FULL_MODEL_CONFIG
            if any((OUTPUT_BASE / s / f"{k}.db").exists() for s in okgp_sources)
        ]
        cov_inv_by_model = compute_cov_inv(
            OUTPUT_BASE, source_df=df_null, source_col=SOURCE_COL,
            id_col=ID_COL, model_keys=model_keys_cov,
            method="ledoit_wolf", show_progress=True,
        )
        print(f"Computed cov_inv for: {list(cov_inv_by_model.keys())}")
    else:
        print("No single dataframe found. Run the main pipeline first.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4: Recompute metrics for survival pairs
# ---------------------------------------------------------------------------
from notebooks.processing.process_epistasis import recompute_metrics_with_cov_inv

surv_db_dir = OUTPUT_BASE / SURVIVAL_SOURCE
if not surv_db_dir.exists():
    raise FileNotFoundError(f"No survival embeddings at {surv_db_dir}. Run Cell 2 first.")

# Load cov_inv from cache
cov_inv_by_model = {}
for p in cache_dir.glob("*_cov_inv.npz"):
    mk = p.stem.replace("_cov_inv", "")
    if (surv_db_dir / f"{mk}.db").exists():
        data = np.load(p)
        cov_inv_by_model[mk] = data["cov_inv"]
print(f"Loaded cov_inv for {len(cov_inv_by_model)} models with survival DBs")

# Recompute metrics
metrics_by_tool = recompute_metrics_with_cov_inv(
    OUTPUT_BASE,
    embed_df,
    cov_inv_by_model,
    source_col=SOURCE_COL,
    model_keys=list(cov_inv_by_model),
    id_col=ID_COL,
    show_progress=True,
)

# Save sheets
sheets_dir = OUTPUT_BASE / "sheets"
sheets_dir.mkdir(parents=True, exist_ok=True)
for tool, tbl in metrics_by_tool.items():
    out = sheets_dir / f"survival_metrics_{tool}.parquet"
    tbl.to_parquet(out, index=False)
    print(f"  {tool}: {len(tbl)} rows → {out.name}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5: Load residuals & build comparison DataFrame
# ---------------------------------------------------------------------------
from genebeddings import VariantEmbeddingDB
from genebeddings.epistasis_features import (
    _list_epistasis_ids_from_db, _load_residual_from_db, compute_mahalanobis,
)
import torch

# Group labels
eid_to_group = dict(zip(groups_df["epistasis_id"], groups_df["group"]))

# Find models
model_dbs = sorted(surv_db_dir.glob("*.db"))
MODELS = {p.stem: FULL_MODEL_CONFIG.get(p.stem, {}).get("display", p.stem) for p in model_dbs}
print(f"Models: {list(MODELS.values())}")

all_rows = []
for mk, display in MODELS.items():
    db = VariantEmbeddingDB(str(surv_db_dir / f"{mk}.db"))
    cov_inv = None
    cov_path = cache_dir / f"{mk}_cov_inv.npz"
    if cov_path.exists():
        cov_inv = np.load(cov_path)["cov_inv"]

    epi_ids = _list_epistasis_ids_from_db(db)
    for eid in epi_ids:
        group = eid_to_group.get(eid)
        if group is None:
            continue
        r = _load_residual_from_db(db, eid)
        if r is None:
            continue
        r = r.astype(np.float64)
        r_norm = np.linalg.norm(r)
        if r_norm < 1e-20:
            continue

        row = {
            "model": mk, "display": display,
            "epistasis_id": eid, "group": group,
            "residual_magnitude": float(r_norm),
            "mahalanobis": np.nan,
        }
        if cov_inv is not None:
            try:
                row["mahalanobis"] = compute_mahalanobis(
                    torch.tensor(r, dtype=torch.float32), cov_inv
                )
            except Exception:
                pass
        all_rows.append(row)
    db.close()

df = pd.DataFrame(all_rows)
print(f"\nTotal: {len(df)} residuals")
print(df.groupby(["display", "group"]).size().unstack(fill_value=0))

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6: Statistical tests — significant vs null
# ---------------------------------------------------------------------------
from scipy.stats import mannwhitneyu

metrics = ["residual_magnitude"]
if df["mahalanobis"].notna().any():
    metrics.append("mahalanobis")

for metric in metrics:
    print(f"\n{'='*90}")
    print(f"{metric.upper()}: survival-significant vs null")
    print(f"{'='*90}")

    all_sig, all_null, all_germ = [], [], []

    for mk, display in MODELS.items():
        sub = df[(df["model"] == mk) & df[metric].notna()]
        sig = sub[sub["group"] == "significant"][metric]
        tcga_n = sub[sub["group"] == "tcga_null"][metric]
        germ_n = sub[sub["group"] == "germline_null"][metric]

        all_sig.extend(sig.values)
        all_null.extend(tcga_n.values)
        all_germ.extend(germ_n.values)

        if len(sig) < 3:
            continue

        print(f"\n  {display}: sig n={len(sig)}, median={sig.median():.6f}")

        for name, null in [("TCGA null", tcga_n), ("Germline null", germ_n)]:
            if len(null) < 3:
                continue
            _, p = mannwhitneyu(sig, null, alternative="greater")
            star = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
            d = "sig>" if sig.median() > null.median() else "null>"
            print(f"    vs {name:>15s}: n={len(null):>4d}, "
                  f"median={null.median():.6f}, p={p:.2e}{star:>4s} {d}")

    # Pooled across models
    all_sig, all_null = np.array(all_sig), np.array(all_null)
    all_germ = np.array(all_germ)
    if len(all_sig) > 5 and len(all_null) > 5:
        _, p_mwu = mannwhitneyu(all_sig, all_null, alternative="greater")

        # Permutation test on MEAN (captures heavy-tailed outliers)
        observed = np.mean(all_sig) - np.mean(all_null)
        rng = np.random.default_rng(42)
        combined = np.concatenate([all_sig, all_null])
        n_sig = len(all_sig)
        n_perms = 50000
        count = sum(1 for _ in range(n_perms)
                    if np.mean((p := rng.permutation(combined))[:n_sig]) - np.mean(p[n_sig:]) >= observed)
        perm_p = count / n_perms

        print(f"\n  POOLED (n_sig={len(all_sig)}, n_null={len(all_null)}):")
        print(f"    Median: sig={np.median(all_sig):.4f} null={np.median(all_null):.4f} MWU p={p_mwu:.4f}")
        print(f"    Mean:   sig={np.mean(all_sig):.4f} null={np.mean(all_null):.4f} "
              f"diff={observed:+.4f} perm_p={perm_p:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7: Figures
# ---------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

colors = {"significant": "#d62728", "tcga_null": "#1f77b4", "germline_null": "#2ca02c"}
labels = {"significant": "Survival-sig", "tcga_null": "TCGA null", "germline_null": "Germline null"}

models_plot = [mk for mk in MODELS if len(df[df["model"] == mk]) > 10]
n_m = len(models_plot)
has_maha = df["mahalanobis"].notna().any()
n_rows = 2 if has_maha else 1

if n_m > 0:
    fig, axes = plt.subplots(n_rows, n_m, figsize=(4*n_m, 4*n_rows), squeeze=False)

    for j, mk in enumerate(models_plot):
        display = MODELS[mk]
        sub = df[df["model"] == mk]

        # Row 0: residual magnitude
        ax = axes[0, j]
        for grp in ["tcga_null", "germline_null", "significant"]:
            vals = sub[sub["group"] == grp]["residual_magnitude"]
            if len(vals) > 2:
                ax.hist(vals, bins=30, alpha=0.5, color=colors[grp], density=True,
                        label=f"{labels[grp]} (n={len(vals)})", edgecolor="none")
                ax.axvline(vals.median(), color=colors[grp], ls="--", lw=1.5)
        ax.set_title(display, fontweight="bold", fontsize=10)
        ax.set_xlabel("|residual|")
        if j == 0:
            ax.set_ylabel("Density")
        ax.legend(fontsize=6)

        # p-value annotation
        sig_v = sub[sub["group"] == "significant"]["residual_magnitude"]
        null_v = sub[sub["group"] == "tcga_null"]["residual_magnitude"]
        if len(sig_v) > 3 and len(null_v) > 3:
            _, p = mannwhitneyu(sig_v, null_v, alternative="two-sided")
            ax.text(0.95, 0.95, f"p={p:.2e}", transform=ax.transAxes, fontsize=7,
                    ha="right", va="top",
                    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

        # Row 1: Mahalanobis
        if has_maha:
            ax2 = axes[1, j]
            for grp in ["tcga_null", "germline_null", "significant"]:
                vals = sub[(sub["group"] == grp) & sub["mahalanobis"].notna()]["mahalanobis"]
                if len(vals) > 2:
                    ax2.hist(vals, bins=30, alpha=0.5, color=colors[grp], density=True,
                             label=f"{labels[grp]} (n={len(vals)})", edgecolor="none")
                    ax2.axvline(vals.median(), color=colors[grp], ls="--", lw=1.5)
            ax2.set_xlabel("Mahalanobis")
            if j == 0:
                ax2.set_ylabel("Density")
            ax2.legend(fontsize=6)

            sig_v = sub[(sub["group"] == "significant") & sub["mahalanobis"].notna()]["mahalanobis"]
            null_v = sub[(sub["group"] == "tcga_null") & sub["mahalanobis"].notna()]["mahalanobis"]
            if len(sig_v) > 3 and len(null_v) > 3:
                _, p = mannwhitneyu(sig_v, null_v, alternative="two-sided")
                ax2.text(0.95, 0.95, f"p={p:.2e}", transform=ax2.transAxes, fontsize=7,
                         ha="right", va="top",
                         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

    fig.suptitle("Epistasis geometry: survival-significant vs matched nulls",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_survival_epistasis_geometry.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIG_DIR / "fig_survival_epistasis_geometry.pdf", bbox_inches="tight")
    plt.show()
    print(f"Figure saved → {FIG_DIR / 'fig_survival_epistasis_geometry.png'}")

# Save residuals
df.to_parquet(FIG_DIR / "survival_epistasis_residuals.parquet", index=False)
print(f"Residuals saved → {FIG_DIR / 'survival_epistasis_residuals.parquet'}")